# ⚡ Google Drive GPU Refiner - Remote Worker Node (Helper)

This lightweight helper notebook provides external GPU compute (NVIDIA Tesla T4 / V100 / A100 / L4) to accelerate video refining for the **Main Google Drive Downloader & GPU Refiner**.

---

### 🌟 How it Works:
1. **Shared Drive Integration**: Mounts the shared `downloader` Google Drive folder.
2. **Zero Configuration**: Automatically discovers the Main coordinator via `data/cluster/master.json` on Google Drive.
3. **Hardware NVENC Acceleration**: Receives 3–5 minute video chunks, transcodes them at maximum GPU speed using NVENC hardware acceleration, and saves the finished segments directly back into Google Drive.
4. **Fault Tolerant**: If this Colab runtime stops or is preempted, the Main coordinator automatically reclaims the chunk and reassigns it to another worker without losing any completed work.


In [ ]:
#@title 🛠️ Step 1: Environment Setup & Google Drive Mount
#@markdown Mounts Google Drive, verifies NVIDIA GPU hardware acceleration, and prepares workspace.

import os
import sys
import shutil
import subprocess
import json
import base64
import zlib

print("=" * 72)
print("🚀 [1/3] Checking Hardware Acceleration (NVIDIA GPU)...")
print("=" * 72)
has_gpu = False
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ Active GPU Detected: {gpu_info}")
    has_gpu = True
except Exception:
    print("⚠️ No NVIDIA GPU detected! Helper will run in CPU fallback mode (slower).")
    print("👉 Recommendation: Click Runtime > Change runtime type > Select T4 GPU.")

print("\n" + "=" * 72)
print("📁 [2/3] Mounting Google Drive...")
print("=" * 72)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n" + "=" * 72)
print("📦 [3/3] Deploying Helper Modules...")
print("=" * 72)
workspace_dir = "/content/colab-gpu-helper"
os.makedirs(workspace_dir, exist_ok=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# Decompress and write modular files
bundle_b64 = "eJzdfYtW40iy4K/kUNuD3W0M1Ot2e5a5S4G7imkedTFU3TmY1ZHttK1Bljx6QHm4/PvGI1PKlFLGVPfds2fdM4Vt5SNeGREZEZl+3BrH0TSYdZerrZ7YChbLOMlEnA4j9TaRxds8DybFhyxYlE/SeZ4FYflsnkh/EkSzYTRN4oXIVkv4INTTi2UWxJEfdsRpkGYdcRyM4d/DaKVaL/1sHgYj3fwzfBxGw+iVOJaZHGeiH90HSRwtZJT1xMc4noVSHMWhPxL3qTiNxz4AcjLwji5ODz+IA8CliyN25TeYLW0Nt3YB4ww6D7fawo8mIoqzeqOj3hBfW22ceiKnQIc0Du+lN4kfojD2JzLxJkHSaoudv4o0S3rDSMBruEX/0fvjVeQvAgAnXIkQwMpkqsE9ToJ7KcqhBAwFqMXJqiPSfIl4I8HSObwZ51na4RG7+oudzE9mMkt3RqudYMKw5UjVDmGUzv1ETsQ0DmHsVDzMA5hzlvuJD4hLHHkK36Q8KDQV5/0v/UvxkAQZEEYBA1BnsQ1wEsdZt45oMCUaaporUuBr5KcSyWSwwR+l+LelP8NTIJNseR7C5Hntdtk9kVmeREXXf8RB1NJDdgAEgxdp5s9Q4Jhh2PmV2O+Kz0mw8JOVGM/l+K7XSMCWhaZuJFQjEUcsYAo09bWH5ADMSonanWD/3YZZDHJVxM0csG2QL0tWxid8TeMEVt5EBESVEHqjEDb1L8bxcDaDCURJs1cHR23XeyrW6n5BStPRcK6J8IW0CKJcOsCooaNfQLskgOVxIG5kN4wfJC4sxFVWMeWpb93DALR+tGrdYSc9JI5CX9yAvPiZP9xCyUn8B/VGToNIAm9u2wJaYnfZldEkfQhASIdb3WC5ikaoKgpw1MhN+ONLCS5D626W5qMaQ6i5Ldgk0A24VqQIRmSFZnMLv34eVGhVbyO/jeUSFC79Ae2yMcef6bj007Rcpq+74giXpxj7URyhwsR3kwC4JQvFGGh1VTwiaSnHra3CsxWt5l2Tlp0XtRc7YqBW8vquA9K39D615uM+SlpRgMYoQCUKBl3q/BxX2eZeP2BqQ0OS8CMp2LF7OVNzU0rUbPi9S3zo+wbGGzBrtc7DoM7AJVd92kR0kPJObahqY2h0+7wgIwB2ow2kmAWyATWmQ40bxpzjUpjfdMVA+sl4LtyYkhSAllHGGVwHn2096huxXQrPNo+4WFF3l5kpaNdkVlTf5y0KarRVRdM6Ohdd8jD0llXtpTp0eDS38NEjreC7hLtWtJbKa2BzswFhZrgXAoPboEYVYH/kqjCG/X9ndZREalgl+rWhzsdXfdU4UGd6KR48v3y54XdakrfF4mONzP6cgjE1lLRrOZlKvHlNmaN838JqGqHo51xdZq8/ZIk5hHo9kxoYpF/gQNSh5sYbeDRObwZgcfszzwJJbb5TiI7l1M/DTExh4zbyLbek9ObUbnCyRi87HAGjV10z8KN2IXhF4/8OVeFsVFEhL8dRY6lUDHfHr48vvp6fXhwe9y+945NLGLBpNz2MAPPKRvL04ujw1Pt6cfnb4PPhUd8GJ5OLJU4tw1Sq9h8OB301zUs3nfWpLHHWI6NAF0EE/MBQtDlMMciSfAwkAMpZ28pj05W9PPxagZFmsAlV7FRg5Mv+ryfn/eMNO+ldjdHx7OLLSX/g6m8MjZ0X8T2CaPa9+rJBv+zeS2VS7Xpx9ang+rrecTZXuuH48OpwMzR5PwddzvpXh9Ttb4OL81o/NR7hJjMfe3X/kcZRwbH+ci4XMoEVTuEj8eXMZhaLRQPLKjKjeOYFPLzqSrhu2huZp2MZV5eHg0/e4Orw48n5xw0HyBI/nVsREW7URFjHEAVl+dla+tqDu6l81T/7rKauzAb8vjz5omlj81iF/cAeJMEoz3BBfb4WR2GeZuA6/xm2jXl0l8KCI0xtnq0hWV2QTFodfbo+/22wjuTGI9IEBIVBLTXCZpQ2eh+dXg+u3IvFpO6Y8be64JD9S+ecxqjY+yFO7mRiT3h2SH9+PTmtqbxK74WPU5d8VQg3A14XDhN8u/saJGrzVFGxG6xByDlSFa2q2GmRO6JYeZ746EUoUfGOry8Pr04uzr1B/whme/12r7tX+BbglIpFEOUYr1jCCHdyNU3A9IhUzjCCLVoxuCQLUDojP/SjsRTxVLADksUh6CT86s8C/DllRX69HvTFye6FllTvtI9W6erkrH9xfaVgeLOnYHgl3hXTh9JPpZgo6EVrmch7gCBFTwce+CPwtTisnmIrXFFvdxYYr0ARBQ/Xj9JxPAGvdRgxq7xP/cPLqw/9w6vK/G/fwfSvxJmf3AnmEqA1DcEmcURRzMFNz0bSz8hNfvsOFuzZ4X96h0dXqAs0d/528WGAFC2JecShpjgHagZLiSP2BCCyM8qnU4x1R/JbJv4Rj7ZTBjsVaUxaQwkLNLjHbXcAW7BJKJnVX2QyioEEp/GMFMmfwQkc5TNxHmfBFDw/pFcqruAp9jjuf7j+6J1efBz0xCiOcc+JjoXktER3JjNwpvrnhx9OQasVbdlR2EfP+0/oweyRoA3+DoieYQMS1h6mEtat/XQFcrgA935WyikoWgyKU6Ik7dk+h86IEPqI2jherrTK/Y/r/jWA+GHdhP/MZS67k5GxyJq61Ze61RlhPVGB9I7wwwd/lYoc4A4JwMHgWAz+4zTIJCUnMPcA7AXxwyighROtAGyzAz0JqakfhOBvpX8R5EU8BDAqjmyj6fItDVxs5CyP0mhVHZLchyiF2UGyxjJNMdtgBC0FbWeGEYo5u/IqOVSxQcq5gDemW1T33VQY0nbMyoaFt9URJSMse1U3b/CdqQ0d2rjj9I47wvKLio+Fr9MRNe9FNyqBq5rM4hsTpkY7MYz0jsXegINgLvw7CYxIW5MOM8GL7w6uklwqX3/NplBvCEnfjPMkkdF4JQZy4S/nMcgZKIivrNVOQUIzpbw0SzV8qIf5yenFV2/wuY9crbfZQ722B6IxCThHeH1+enJ2ctU/xsA6z56hAdhJlxJskd4yqVkVucvxGvQo5TFVV/DTFSrQvkjWdgsEWy50gGwAhEdAeM6RzmPQ74AM/u2JPMLli9ZzbBAR10GKyICxg0WeBOO7UJo4QWNQ0CkwZCMobfSVmrkKFjLOMwoeDIhoX/1sPJ/EM3GZU+KzQA5k87JqvX4B42k0Kbl39emyP/h0cXrsffiMpN7f2xM/wr+v33aJi/j5tw+7qbuzNcf+a5rEIjTDctm/uoTFznYPVWaQBaAff4KPsLNWabI3oB8zDFlmuPXMUq1edxZyEYMGuvLTO3GV+Eo/AtEx4reDSXsRohJdxDBunFCOPoPGlE25Ve89VKwW0cGI3OHuHB974ziP0BsC2eV5P4bxSNkY+Bp3xkkcpoa88RPPH2ccy8ZlOIx4l7rRw0aAODmf+kilf0naztPOHv8ha+pM0A9U+1ToDpydSeI03VmGfgafFoDIAqxbMArCIFuRMNlpYn8qs1XXkQmnyWtp7OHWfTCRsW5Ls2IkpJvmo1ayfYP1Brs//ntvuPU///pft9sdgrVD7druLsNh+hO1E0XDLjqty1bbCsXgk5seiNwtQggIE3na4q8ohgINHTXR1BytUAF6vI2cyFBmskVxErT1HixMigGV1EUnyFH/8IGGqRY9XOGo4kOg9e1ZjPFYMNts9qkCxOoQROAK4PNQ781lsV1H6UiX/hg7J2mm7BQ0jtDZXoCKibCOIYFFgbMEGQ8Srrriah6konCBTdj+mcPSQilO8wVZhuaShzpVMInsKCipN2zXBeRX9MOfs2YOk+qybvjKowA8ME9JzVQHBh6xjqeL/7xttbtz+e2m9/Ptk/eoQUZnktaQA+gnTQB8ES1BQNKs6go6gDSAMUB8RfzHzYHN9BiZzgynUhVye8tuXG/URa46oOwYkFlzXYNJgp0MyRgPPgnSu3U5WGOgStAYGrJYtZyTKZayLnO5G8IH39D0OUAXZy1g0g1JIS+eWzBcCcaCeuCJ5CHXKvHyFMRMUluPdRI89cSjfLJC7fUMRR3d9WJaR93RvjFJWpLCRQ5nSL4yglofrKTGsJ2N8iXPSjUsHugjf0UmAJy+4uteUXB2Aw1ulZ/SdiiswRh2uPWKJ9FyB7+5JIQmFbDJwZVPEQSruGo8Bp0PZhOLwHR9Fe5RJLCxLDQqC7hEkKY5qCoYjPY5s3kMywt2bWD3lXqjCi+wPYZigxEXsONBW+9Q3m79VVII5mTfTVfDOSq5mAXVnLQxBgC8QYbaoRodiWY9mUsXlqVCjmT1s569PfIr3DODJxOlBisnej+N1NCRTJOnLGZL3dOqQgHTjF4DNG4lw63/zRvgH4dDkI/uj/8DrTS0OPl4fnHZPzoc9Nudxq5FDBV7o1it639rpK8QWrBrqUzuOdAzklP0odWC6emGJ9Niiwj+N1AUeTP3ga962448tUK5OHs6ziWZ4xHq/dJmoxC4+IV60ZvUap0awwS6n1tY1GhVrUTLvz5JUZLQMIeap5rz58Hs3CNWyoHDWD78q9hzq8bCUpcoNhlo/VLmDGMzU+I/T9LRxHN0Kc3FEfP1VlwSbybi3g+DSclE4o6hzcCwPqpxn4xyyYJXyPGXpVA2YBl2cjLNNZvBtubJ1vCOJmvmHj/+b+cfTtMpSfo9PLSwb2YkNitt/eaORjGrcjK4SrbcMNJSJ8VXehM8DukSygzv8RdWsYNS0cacXAfgqEFtLBqqFZ4SYYtxqrJUL0FEiKAlwmMr7LoTBE+A69l43mIoXAViLlZhohSrtMlsEEuwlmsKG1mkFw71VBNZfDn3VwVm9faa2j8diH3HaLAlvisMrG5ri7cD/CPVMF+KR9XpSVvA0W7pxzh2ZFwGrmO6l3nEyCsDg9XSi3iSQ3NVwY/FDHlEEhaoALArFvuMRwfTgfnbCuPZTCaNxxVgp+A+ooD1ltYX6w8pVI4mdAonUkWYqPMOumGwk0AZiBN/hpkaFf0YRh58X49bXOrABTB+lM882GmNQjkpMgmoYdA69/l79BInuiAF15if4hfQExMVHaxNAUMMqgBTExPcR8HIiRyD9OH8sHARkRvEA71fwuX2VsV6PI5NfvQ+nQyuLi7/zokjBXnh2rv95yLyIjPGpGVhYrvXjAy6E7Dforf3KuNCXZGClHfxKQAjwFlWEpRa0ZUZh5lsyvEjqp4sKG5Ivt0YgEfoNKzGWisTODfO5M0tlZ3sD7dQmiuDUgCFMjprd1ssWHzcpqa98AhOOR9M5sISXxvWMiFvglSxxh2luSRfOGWRA6zcPIG9gYLB4oXaklWhLIVCy1DLIUO2dOhHsJ9J8bSFj7GbFMQW497kYJLlQPFHkRYt2Z11UcuYSqntlBRLkJ+TFKsxcGCpDh4hRh4GBP3MQw2SZv5iWT34o+ihtUy3eBPFD1SHl0zxY2u49cOn3g9nvR8GxtkiD1Vf4mHIKYb5MavZsmn0NViCKsa64aWMUJGI4RBPgowx4MrqC1xysG+YMAsFZVpJnwJR1aq6GAgd2bSXlS2qHuo0D7eqCCzMAiL/E4cYxY/il3f0gb7ePI9COBrjZvJbxuHDSlSWNrXUJkX3BjZ3cQ4K2M+zGKivzlIlEvAYIw3ySEbjeEIa5ToKMDkNPoyf0ImdUM788Uoooq5DGWxGl6fqlvC1nc+nYZ7OWzbmLTV1P6J/kyQGPXsxoDfrS0QBepAzcE7BNUla5TTo+RJmVJPS4YAF77DzbLrzs6loCEAkLQKNpgb+dKmzBDU3Bs8K4UgPsK4I6YbVf92J1M8r7kaNFMXQzQ0tmrxQRYXyXoZeulqMQCvTB1fQPrxH00iPu/lyKRM9G0az8Rnq5pPzXy+GW66w+zB/vf/ml2E+lXtTTToZmn1J8zZ1nvz8ZgL/TvYmtc7gzsAi+Xp4ec7pfXx3cv7RrmA34Xjv762Do395eXHZiMS/vR27uw2uj476g0Fzx713Fb1NX++9fs1Vm7Q+5SIgrW2woSMyf6beLTC3PZPqE2iRMShKdCsL14XCc6642hGGH7Q9wWmw1oYfHS5Rn6Vq7wBLPihSWFzR0THUATpaZA8CMjF81pFgoMdqpTuiXY06X4fxS2xoTKKAJRlFaKzitlV2tFZ0CV8ZxoccxsOOZ62g0aM91HCraA1sFcUR0PI5AYnP6E3tMXCOOvqz2iPFSnys3tZ7o6eM3dGEkeEyQH4yrIXha3Z9YmbLsZXDlJPZFLffVe+zQktr5GW8bO1ZoQrAy+NaGdzVPMLHp1t2zuCt9shMPck8yMgH5C7p0614rCug9pN4VKPDO0Wgp7K8GV+vaPuUgJmZxdoYk81VsTYSJXwQyQcSW0ozGWRzGXy3DS7hZsMbVWI26I7a/rpNR/eZGTPOUSuKtoZrPx/+oOUVL5HHVlfUiHTeUtuyg8J+tTEqMW0o4p9216Futtz4aBfruAjruVYtt1KzNdenYDbfATbGCeZ+I6MQTCxwO5phvLylqpiI3RLPYc/8ZAJPU6ziYz+at2a2e1rqWm22SNMW8Fj6VVNdudb5GKuMNsJhwG03hL4ZxMK8bAwlbzs2gfEYnJ8AN1GTwJ9FcQpOHi2YlqZpjC5soKlZ7kaawVUqex2wbKo0tA9+Em0ErI6T0Wq2yboxMbWzsCElyXvbCDjyN38XaMr72Bg2cFxNLd2iep8eJuy5Noa8OFf8wbkN5ZFYW6ocD+iItDxMMYFBbDe+0a4rLwfTQ5bdudlhGG+bcCBX141ItYZEw11qqUUeZgGZASzCADmJpw6sNsOgTHM54a+5MPWYq0GH4dY5e0+RVdAKA8fJBABfSYznlV0RB10QVCjlyvH3qpmx7bG82YZvtm+1UdafXZa5mFE7EMUI2gPCcSqmGh7T5+1b217D90pwoZNlL0rPFw0Jh59p2raZUk5YlO2lRcY+NXxTpOU8wKDfajN2Wt4MzaPjmXMZwoZG1Tm9PKxJeRHnBS2VO1nyJAyDUTeR/8wlFqTaX5OaqX65BKSNO17y0TKJ0aBsGEJ1BE/5ypdhCbGOhqmPHNw11+ZsmXtBNI053mLrEYtHJ1G6lKAgBNeKiDlY4ge82wQF9/zLyfHJIdWBYyt7DeLwtgM+3MJpqSQK2AH8h35c8Fm5B+A+8RfeYoSt9qwHFMNOuLee58kVe8AyxQODtl3KwXjw9TLPWvYiuRluRfcBGMqddBHwhnNnB9iZrHYA3gOEt8MC2qXqwA5D4al6St2DFdbBOL3vRPFc4uE7eJNHWMi6dVvZCqA+PLBLNyo1Zvha4ilTVBhL/YzzL3T0Nc+66TIMsCq+g0cMazsD6g1bggPxphozBdbcmNy4pbgcNL/Zu3U2LTiCLTHpwa33b6upQ26u+VSO+/p2w7CWoVNwMCPoGoENHnv+PTg26Ko0hF/pzosUaXD+pX9+JObyfrw7f/3+LbvLeFYBxVddB0SeECw4cICUqmiOZ62XKcB6Ol0s5UzJg54NeV+yu4PBN9AKB8Ywx/0v59enpy6disAz3qjviekcr0KEKg+epW+13GccYr3TJ1KUvAwdQYbDHJzaeIEnQ3CZX1L5aKLrtCMMDZZJqDSAj766NYo1MNpGCUy6U/GD4yAdx7hyxJkPcN8Hvr5W6ZgzlgBVsEgxf5bF4rVZq01lnur4iVpOTOSUhik0E/GdoxiJciawAgTzgRN9Pii1bmMyIXDEOXBHfXRxfnR9edk/v1JF9VhIrDeMFOgBfQoun9dKZTjtKGnygsn6Oil8YYeuVsgcuDT0s9EOy5rwCUqh2cdeyl0VkAS9wPqZBRLjk+WTK5mGfvlAaxeSrXMgqelI0FwFPjB5+R7aT/WBMe+xAFCfk39qKIqsjY5yZBRUskyK1qON5rbGEnyQdm0MPmzmgZFtSq1ZzZXUWhnCShMunfbSMM7SHhtJUH2l2a363zjT416PZuuIfX7ztGbQ5vprqwtXQmJFeuXSL3K2dJybbT5ytHoUlNhcOdrd7pB6KZwky6czwxgOCOqhi3IZTNT69kp20BCkrC22GCtAe/7+JBXcT4ykP1a5ck7EqDVqJqcsXU36Wl0sZtWO4atOI8eRRqSTRVmdP3Sdsqwfb2x3npuz8TBl08xrTnk65i+nr+wvqFpD0ebZWlX3TWeNN8SUQSpKryTfEZci0fGpXokK8vBoQ2vacLEMyBMWSGL1Dp8ILOVs3Z1e2I0KPIEh8L5yg8c8y5bN1+PgS9lO6NpNWFnyxQ71Hhte9OKsxNUVAEQRibeuoQM5ll4Wc/mGMi4AfJan+lTjcIuOXFrwq8AKx/jHMew70FXdmSZS6vMk5gpzGMO1C02xqxZcLywDuui23ajFwgudX7Qtvqm1NfcNa0xfrZ+xkah2Kz3aWi8mL3Wid/UcAcicV5y1rQT0a60TOcOsOxDVqzV2Rv+/Q42ZZ/deqsbK894vVWEbzPqMCjPmblZfaPQaNdjzgfjMabKqPR4owF67O5D6mj6OJdRP7gJKGq9QjDwyYfsd2pFU4iRfLFu46ACWDp5pBEN+8Pr7o/b4npxW1HyeCl4o5SKjyTKmBCR5Njhrr9Hl0e6sZd3dYUl8qQA65Sb9SHy6uvosPl8MrtCR/Ni/EgoSJt4oiR9SUFS8l6ZcpH8fB3htQJxPpiH6+iM8uKHOK9e0lIrhVb1DZyzP9hDZygDbHyudnx41eSwnti6EgImntKQ9NL40RjUNyjS6Bqx3Dmd0DQ5GOs7ifwVh6O/i6f/W1wDP/6Xi/Ers73X3/iLgi/dv/yK+vX/bxrxvKL/K0W9Btvvuzb9137wXrd8+XZ2ddkQY3EnxETascVsczcGjkrt0WBL/EwN/6ieB6mIFY0qoDscoVwyRD/OoeOcurwBnl//c4X3izslx3SDYHZ7sj5iZQOoFXD9PR1/rMxhELpZJSuukrWs1ykVW767YAMbgiH3jnavVUuqitDqONQ5DOzv+171USwm+5qVzoGHs6OkO1N8KQCTyldHgI2kQ+NwhqwHb/IM3e6QvEpkuHSThG64wXoSWC5Ptr/dcFdn4GsWTFV8atcRr/SatooJlHdUYd1o0hbuWtnCseq2KGQbt4mrnbEqldFrBjaYB9v4eHoYdbunr/Ap1VIdE1axSngk2jf3Tz32yQ7iDRM3yiEc/JvIJQzzlusUiYkQXYK/XETu1gQuZ68vT78eFXFGuBphQ6KZogf6pbOv988twBiGOwIsrTjv9QWirU9+E7Sa687lC+YJCvwN9ToRW8C9nrCG+IcYNbrh24Dxco2pjW25qjQgkvvRelrvosGLpYdM6L2JfJKeHn09IIrgM/AP55TVjBkB4S3+Fq+3/G8/b6f0qFquiggqPi7KxsdwXmsjFPa9E1nEcJxO83yQuIkiG/W53u91K/irV8TTbG4INnr8MdpW7uquFgA+LFbywC3ISqialm+PV9lTvJdpc91TuBayqMnyBaxbM+PRH2V1/qejcqbCrItSKdLqYoUI7VbeARyJWogSkgXSwarcf9fRP23+yiGZc4lKZu6aYhlt8fxiLOk+rrkpSNbd/EekKNrkTWhCVG+NpKzrSK8Kq0OFoWn2HTAQ/KDbFtdXNgb5ydacARbmZU07wAq+EGKebOb5ODcBOLl0KsUziWYIVIyga5RVN4M8i1WsrvVzlCgr0jh+fag1urFV/q6XY4WBtKuPGlrYj6gK+nuKqfUXuO2V4wl4q7uZqmZRVum4pc+r/Ujf88kZ80qj0+Do78erRnHGb0ghApG/b7SdQE9az5Tjb7gjwi9u97v706Ye2+F/CbjFdptgCusI7l5DRcYhntUF8Z8g0p8q10+AImj63rfoch6FKq+C2ma7sKrJl6rYxXtCYAamJXeGc0MbHkosSrH8vpOuguhX+c6GVDh7NTHcXL37gyJWhuJ5s29Ygnxqm57Ts3E+Vq1V1Gja1Ju9gV+mTQ6LFRY9uiAqyXPwk9p92y6eUEvY4K4WyFE/F3+KR2f8f8Qh6k5xtG/2CLJQw4ja0hmfb7SYXJZHPBwp1fqsUnY7gaancpiMKHNRn8LoYZnUWiauPscIQD3SCde6JKYg7CgOug2Ytl4/wpiYlXQoM1O1I8LHcQMf9Lk+GccSG/K7WoMAb2xQfas1scmBb+5taB5NU2Nz67PJqNlXBFifdangzHSjfqnv2QKRrxh2EdKzEvCDKk8m+FsswU/WpXXWZXqDjfAobqEtzS4nl7TmTWAksYtyr1KJ08FarTMsxSmOzLJaf+9/kmO5m1DnhHYQCr33EFDDXBRT3LtKhluKuPDxsXJCnY9o+Ph+0BHcTTNO4tONph2/G4cskBF4ym85hmu7i7t4KrJvpZRfUeL0tH9Jk3hwQRW743ltTOg2/GcXU2acq0bcm+6beeDpTbRXX9O24niodAnF6fDJ4zpJQjq9XnjFuIUllK2P53Vaqqx1H2isEaFDlXLpZkfubAYiJeFTC8nQreLsqBnGejLXxK65joPmQ78d849JjZebG7WJRO2HjrDLXtZt+qgiZaxgTqvC8GnB2plsBwSCilHGBofdYTm2Dy525WGWjoVFUnhnbDshP6YBOgUKnnLLKMsdVOC9LLxp34vzeXFv5jH7C6ggXOjJA3xhbP5Bd3pUE+0fUDoY6qF5NYHO6UzK4bc/8uis+5EE4Eb/+isVChhoC7bsATWrMASPQ1Wt8KZxatlrLGg/Zy45gX2cJwjiZ1rrBd969H2KPN2b+4N5DGMa19lRzw894lk/d1+/fiV3xqf/lyL6lIvWoygp1PvzlsiQ9LNWVHGIP49uyL7pvVMyExWWOQq+K6lBT1WWtGMchCPeen8w4p7a1M+7dMzpmHRWVbdEVL+r6+eVb9eX4n9wew1NAwervZDl2w80zgl/8jejDIydTc+AaCPcyWaH0WYrWMd/3IW9Ul/3fRP7d9yBvyPViwuNa9XZ4yGES6GrMlUWwV+oOc/Hl8vCsLBSjYDcadQw8nH857h/1ystxU4HlqUWTUZyBvx7JsX2vWRPRAUbQeBmWXSMB5g/kgDBw43zi4w+jWRiZrakItVAgt/bmx9IKdOUxL/z1ANwzpY2+jfNbw/zIfKz5vzt+PeeD32J+zovkzKcipyzVhSp30vuXTGJXHx0dUWIXLGVvv9ausDDl17f2HTt54qWkwGjf0jJcG/UIx6dLxNsVlfymK079PBrPtU5WhZkcGKMIThm0URd8bhyufP3e38eb9PnXqAync1psNEu/CbeWZGEt7xu3jQU6at+IG0YKczdXyImW7RPVIp+Ipl3Q+pmyTSANHXVG3CxT/Xzyub+mfNUqch3lU9wMHexbpOayiREd2EQf0VUGgcWe3nKsNp6V76fL1Pk93+B7QGXY9olC9+8H8Y3pEVFAHbd25Rqw1QH9qZdm6xfWWnMDo54I4Dz43p/bMhBlSebhVaU3Dnuzf6vhaUjP0SzsgVN8CmODelx3j9/981R8crpGCeLMGlpY7FuHaQMpGUkawECTP28KI/q/KIZe/v1cw1Qr1cW7cFjDJQaX9BYMsSv2vb29Pfy/KePOTrxGwFa1fvmlu9cRC/9baw/ftIoxd+nLffxS6cB2my5WVkqwaXxFVZjBoCl++oOEzeixIefw9Yf8gloUPzRpHvMFctLCpjuWzqKzFFh0sWZq0seVlIKjyMN8PRvGqnfYKKxV70Yc7RX8fLY98bNXcPPZ9oqbPYOXz/cpiu3MsH9zr6c1grs+MkamuBYoaLDC2ivpiUdNLk4GYC5AE6TX3ZtSCkD8F1+IrlsT6u5bygpYbWMI0rZGYXEoTcN0AOJVc/j0y1AO+81qpLrEHU3JOD74oMjW3EHiSu5Tx7sgDKur62UBnGIK9aMpDYzSMrM2zV+L22ACCsHk5+SR/Ql/M8B9zXQZ38AG1dsPjafApqpyWIc0Xz7ShFnpLE75LPcL/T1xA+DjWcAJhrgq6OIJ0g1pRT8ASeGS8hiNCqg1/KK5EZ9ccyFCJXa55kaEarSloHinGgC1Y1UqIG8EwjTPqt3AVrbwBwfU7w7Uo+0NOXW+BeZ5DmrKidajhorUiTj78HI/Xgx8/P3AMqS8VdnRvOty7H9Vi/ybAW2Y0M4bKeNjGJQqeTsFTSszvu/yzYdGJB1jjHGCv9tiXPJLbHlRCNHtd20YWFQi2BBCfGHBrLFGqjUE+vcUiEdhHC91ob6RvHCU53NpEJ/+4V8wFnjZNCwjOlKLbG+BCyde03v+pSt40q5Xuk5gX4/XlqXl3aH4ephjTbN13qkWN3q+RJbYgC5TGkq5bL1x3aVZXBRa4R7+YIbKexnJ7fqFs5T8cfC5AbECQdyiu89VNYhDrfGNYtKtSlvUuz1XaVK6La5LbLGzle3CSQrROFB/myrI/zjczJzx75sGg15+tCp/WgQ+tFKzVpfv/aaKvurA3Xs/zPHeUfdmmm/LNsdfi+sLyn/0y5Dkve67qkpwRzMNMXTfFotRaUNUoQ16CsZXP4jXey73QL/Wua413+gExu2Jr+Cb6VuFyzoP85RsNeLjIMEbKysLSoILT2q6igyJqmWUlF4NKG8aL8GTxeAZQmId1y1/HY61V01pNUfOaOcMpvj9z3VD/Ewlh/x5T50/xvnVQVI80cpBOHLR7VKUavZszSzwXOnsk2M9TlkA86JxALreeqtffajKJvHZ2QeKZLdfNCOnvT/p4Pehzoij4ROP23+X6bZKrtQP2tP1ItvnsWhhUPxX9RPO7e2X4Wz86pfG3Xm0+qn4jcEA737xIxnnabjS5u9Fc/IvDp7GlV8dRIz5Otpj0VK/U9gm/NWAxiWvjPzxyUA1P9NXPaV0i9LmVLDEuuKJbaLJTIeLpNlecOZiK2hVRkLJy3BdP0N+BqxucFZnqk7LyZaaZ2adXOarm1t8SOuAC12qXlFHYDLhoIVfdNp4QAIcs8h1IivjEF3VUVBI6BttMosozzo7lPfVFxAIuot9CvtpR/Hr9eXpd/pJlRba93Edi3YbwOenWC9kWhmO4d/p/l6Bry4DOirR1IvQKI9uDFkwWV2F8K68vNuKGoA7yoRhK/Vm+sayajW+qDOr8D/HKVX0lYsD0bVsRw2ZNSvugWHACyYVlo1DGaa06kvoV5Nv/Ep8rhUT8fW8Pt4uTL4VllZxZo5/Z5Z/UQVv8larHtSidj955dtzvMy5U22KH67LyuPjzX4cee+m81e9QWZqjeuYllCiWZ5pWPBu8wCrGVz1M5UR1F9Vz7vaHa0ga9HX+HZ9dxVsLTrSZy4rXttPBV2LfvR5g35F8LXoqb7pqLzYM91fEIetxmAb1rubVdZMbNuq4xmLan+vuEcMlLqnbqxYgGJQt82rm2P5Vn7jLo/qVfl4HSzXN6O4KceQPcXyihrDQ9Vxn8pF/MZ0Cmhldg+sm3S0VrSjF+qKIfSy1XMVgvhNrkYxeGYn+AOQSb4005Hr/InS7Z3sa5yUhxroofjXDvIUxhCDeZ6RdsWfkKxqSAXdGr2ofmxaOSN4RSq5q57H8WmP+OJ5RWy6yjG8m+3p/wDsapal"
try:
    raw_json = zlib.decompress(base64.b64decode(bundle_b64.encode('ascii'))).decode('utf-8')
    files_dict = json.loads(raw_json)
    for fname, content in files_dict.items():
        fpath = os.path.join(workspace_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"✅ Deployed {len(files_dict)} helper modules into {workspace_dir}")
except Exception as e:
    print(f"⚠️ Notice deploying bundle: {e}")

print("🎉 Setup complete! Helper workspace ready.")


In [ ]:
#@title 🚀 Step 2: Start Autonomous GPU Worker Loop
#@markdown Launches the helper node. It auto-discovers the Main coordinator and begins encoding video chunks.
enable_debug_logs = True #@param {type:"boolean"}

import time
import logger
logger.set_debug(enable_debug_logs)
import helper_worker

print("=" * 72)
print("🟢 Launching Autonomous GPU Worker Node...")
print(f"   Debug Log Notifications: {'ENABLED (Verbose)' if enable_debug_logs else 'DISABLED (Quiet)'}")
print("   Keep this cell running to provide GPU compute.")
print("=" * 72 + "\n")

try:
    helper_worker.run_helper_main(enable_debug_logs=enable_debug_logs)
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user.")
